# 00 - Pobieranie wideo WLASL

Notebook pobiera brakujace pliki wideo z datasetu WLASL na podstawie URL-i zawartych w `WLASL_v0.3.json`.

Typy URL-i:
- **Bezposredni link .mp4/.webm** (~14k): pobierany przez `requests`
- **YouTube** (~5k): pobierany przez `yt-dlp`, zapisywany jako `{video_id}.mp4`
- **Flash .swf / inne** (~2k): pomijane (nie mozna pobrac jako wideo)

Wynik:
- pliki MP4 w `kaggle_dataset/videos/{video_id}.mp4`
- manifest CSV `kaggle_dataset/wlasl_download_manifest.csv`

In [1]:
from __future__ import annotations

import json
import shutil
import subprocess
import tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import pandas as pd
import requests

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

print('Imports OK')

Imports OK


In [2]:
# Notebook is at szum/notebooks/WLASL_EDA/ -> 2 levels up to szum/
PROJECT_ROOT = Path.cwd().parents[1]

WLASL_JSON = PROJECT_ROOT / 'kaggle_dataset' / 'WLASL_v0.3.json'
VIDEOS_DIR = PROJECT_ROOT / 'kaggle_dataset' / 'videos'
MANIFEST_CSV = PROJECT_ROOT / 'kaggle_dataset' / 'wlasl_download_manifest.csv'

MAX_WORKERS = 8
OVERWRITE = False
REQUEST_TIMEOUT = 30      # seconds per direct download
YTDLP_QUALITY = 'best[height<=720]/best[height<=480]/best'
YTDLP_NO_CACHE = True

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WLASL_JSON exists:', WLASL_JSON.exists())
print('VIDEOS_DIR:', VIDEOS_DIR)

PROJECT_ROOT: c:\Users\Magda\source\repos\private\szum
WLASL_JSON exists: True
VIDEOS_DIR: c:\Users\Magda\source\repos\private\szum\kaggle_dataset\videos


In [3]:
def check_tools() -> None:
    missing = [t for t in ['yt-dlp', 'ffmpeg'] if shutil.which(t) is None]
    if missing:
        raise EnvironmentError(f'Missing required tools: {missing}. Install with: winget install ffmpeg  &&  pip install yt-dlp')
    print('Tools OK: yt-dlp, ffmpeg')


def classify_url(url: str) -> str:
    """Returns 'youtube', 'direct', or 'skip'."""
    if not url:
        return 'skip'
    low = url.lower()
    if 'youtube.com' in low or 'youtu.be' in low:
        return 'youtube'
    if low.endswith('.mp4') or low.endswith('.webm') or low.endswith('.mov'):
        return 'direct'
    if low.endswith('.swf') or low.endswith('.html'):
        return 'skip'
    if low.startswith('http'):
        return 'youtube'  # yt-dlp as generic extractor
    return 'skip'


# Per-domain request headers to avoid 403s
DOMAIN_HEADERS: dict[str, dict] = {
    'signingsavvy.com': {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Referer': 'https://www.signingsavvy.com/',
        'Accept': 'video/webm,video/mp4,video/*;q=0.9,*/*;q=0.8',
    },
    'handspeak.com': {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Referer': 'https://www.handspeak.com/',
        'Accept': 'video/webm,video/mp4,video/*;q=0.9,*/*;q=0.8',
    },
    'aslsignbank.haskins.yale.edu': {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Referer': 'https://aslsignbank.haskins.yale.edu/',
    },
}
DEFAULT_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
}

def get_headers(url: str) -> dict:
    for domain, headers in DOMAIN_HEADERS.items():
        if domain in url:
            return headers
    return DEFAULT_HEADERS


def download_direct(url: str, output_path: Path) -> None:
    """Downloads a direct video URL with requests, with domain-specific headers."""
    session = requests.Session()
    session.headers.update(get_headers(url))
    with session.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
        r.raise_for_status()
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open('wb') as f:
            for chunk in r.iter_content(chunk_size=65536):
                f.write(chunk)


def download_ytdlp(url: str, output_path: Path) -> None:
    """Downloads via yt-dlp and remuxes to mp4."""
    with tempfile.TemporaryDirectory(prefix='wlasl_dl_') as tmp:
        tmp_dir = Path(tmp)
        template = str(tmp_dir / 'raw.%(ext)s')
        cmd = ['yt-dlp', '--no-playlist', '-f', YTDLP_QUALITY, '-o', template]
        if YTDLP_NO_CACHE:
            cmd.append('--no-cache-dir')
        cmd.append(url)
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            stderr = result.stderr[-400:] if result.stderr else ''
            raise RuntimeError(f'yt-dlp error: {stderr}')

        candidates = sorted(tmp_dir.glob('raw.*'))
        if not candidates:
            raise FileNotFoundError('yt-dlp produced no output file')
        raw = candidates[0]

        output_path.parent.mkdir(parents=True, exist_ok=True)
        if raw.suffix.lower() == '.mp4':
            shutil.move(str(raw), str(output_path))
        else:
            subprocess.run(
                ['ffmpeg', '-y', '-i', str(raw), '-c', 'copy', str(output_path)],
                check=True, capture_output=True,
            )


def process_video(video_id: str, url: str, url_type: str) -> dict[str, Any]:
    output_path = VIDEOS_DIR / f'{video_id}.mp4'
    result: dict[str, Any] = {
        'video_id': video_id,
        'url': url,
        'url_type': url_type,
        'output_path': str(output_path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
        'status': 'pending',
        'error': '',
    }

    if url_type == 'skip':
        result['status'] = 'skipped'
        result['error'] = 'Non-downloadable URL type (e.g. .swf)'
        return result

    if output_path.exists() and not OVERWRITE:
        result['status'] = 'already_exists'
        return result

    try:
        if url_type == 'direct':
            download_direct(url, output_path)
        else:
            download_ytdlp(url, output_path)

        result['status'] = 'ok' if output_path.exists() else 'write_error'
    except Exception as exc:
        result['status'] = 'failed'
        result['error'] = str(exc)[:500]

    return result


print('Helpers defined')

Helpers defined


In [4]:
check_tools()

with WLASL_JSON.open('r', encoding='utf-8') as f:
    data = json.load(f)

# Deduplicate by video_id (many instances share same video)
seen: set[str] = set()
all_jobs: list[dict] = []
for item in data:
    for inst in item.get('instances', []):
        vid = str(inst.get('video_id', '')).strip()
        url = str(inst.get('url', '')).strip()
        if vid and vid not in seen:
            seen.add(vid)
            all_jobs.append({'video_id': vid, 'url': url, 'url_type': classify_url(url)})

all_jobs_df = pd.DataFrame(all_jobs)
print('Unique videos total:', len(all_jobs_df))
print('URL type breakdown:')
print(all_jobs_df['url_type'].value_counts().to_string())

already_on_disk = sum(1 for j in all_jobs if (VIDEOS_DIR / f"{j['video_id']}.mp4").exists())
print(f'\nAlready on disk: {already_on_disk}/{len(all_jobs)}')

# If a manifest exists, only retry previously failed jobs (skip already_exists/skipped)
RETRY_STATUSES = {'failed', 'pending'}
if MANIFEST_CSV.exists() and not OVERWRITE:
    prev = pd.read_csv(MANIFEST_CSV, dtype=str).fillna('')
    retry_ids = set(prev[prev['status'].isin(RETRY_STATUSES)]['video_id'].astype(str))
    jobs = [j for j in all_jobs if j['video_id'] in retry_ids]
    print(f'\nRetrying {len(jobs)} previously failed jobs (from manifest)')
else:
    jobs = all_jobs
    print(f'\nRunning all {len(jobs)} jobs')

print('Jobs to process now:', len(jobs))

Tools OK: yt-dlp, ffmpeg
Unique videos total: 21083
URL type breakdown:
url_type
direct     14212
youtube     5135
skip        1736

Already on disk: 12077/21083

Retrying 7270 previously failed jobs (from manifest)
Jobs to process now: 7270


In [ ]:
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

new_results: list[dict[str, Any]] = []
total = len(jobs)
done_count = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(process_video, j['video_id'], j['url'], j['url_type'])
        for j in jobs
    ]
    for fut in as_completed(futures):
        new_results.append(fut.result())
        done_count += 1
        if done_count % 100 == 0 or done_count == total:
            print(f'Processed {done_count}/{total}')

new_df = pd.DataFrame(new_results)

# Merge with existing manifest (update rows for retried jobs, keep the rest)
if MANIFEST_CSV.exists() and len(new_df) < len(all_jobs):
    prev_df = pd.read_csv(MANIFEST_CSV, dtype=str).fillna('')
    merged_manifest = prev_df.set_index('video_id').copy()
    for _, row in new_df.iterrows():
        merged_manifest.loc[row['video_id']] = row.drop('video_id')
    manifest_df = merged_manifest.reset_index()
else:
    manifest_df = new_df

manifest_df.to_csv(MANIFEST_CSV, index=False)

print('\nStatus summary (full manifest):')
print(manifest_df['status'].value_counts().to_string())
print('\nManifest saved:', MANIFEST_CSV)

failed = manifest_df[manifest_df['status'] == 'failed']
if len(failed) > 0:
    print(f'\nStill failed ({len(failed)} videos):')
    display(failed[['video_id', 'url_type', 'url', 'error']].head(20))

Processed 100/7270
Processed 200/7270
Processed 300/7270
Processed 400/7270
Processed 500/7270
Processed 600/7270
Processed 700/7270
Processed 800/7270
Processed 900/7270
Processed 1000/7270
Processed 1100/7270
Processed 1200/7270
Processed 1300/7270
Processed 1400/7270
Processed 1500/7270
Processed 1600/7270
Processed 1700/7270
Processed 1800/7270
Processed 1900/7270
Processed 2000/7270
